In [1]:
%load_ext autoreload
%autoreload 2
from omegaconf import OmegaConf
from oat.data_module import factory
from oat.model_module.model import OAT
import torch

In [2]:
from torch.utils.data import DataLoader
from oat.data_module.wind_dataset import QueryBatch
data_config = OmegaConf.load('/scratch/mch/cdonner/observation_assimilation_transformer/configs/data/wind_data.yaml')
dataset = factory.create_dataset(data_config)
#train_dataloader, val_dataloader, test_dataloader = factory.create_dataset(data_config)
#batch = next(iter(train_dataloader))

loader = DataLoader(dataset, batch_size=16, shuffle=False)


/scratch/mch/cdonner/observation_assimilation_transformer/.venv/lib/python3.12/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'cfgrib' loading failed:
Cannot find the ecCodes library
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


In [3]:
dataset.stations

array(['ABO', 'AEG', 'AIG', 'ALT', 'AND', 'ANT', 'ARH', 'ARO', 'ATT',
       'BAS', 'BEH', 'BER', 'BEZ', 'BIA', 'BIE', 'BIN', 'BIV', 'BIZ',
       'BLA', 'BOL', 'BOU', 'BRL', 'BRZ', 'BUF', 'BUS', 'CDF', 'CDM',
       'CEV', 'CGI', 'CHA', 'CHB', 'CHD', 'CHM', 'CHU', 'CHZ', 'CIM',
       'CMA', 'COM', 'COV', 'COY', 'CRM', 'DAV', 'DEM', 'DIA', 'DIS',
       'DOL', 'EBK', 'EGH', 'EGO', 'EIN', 'ELM', 'ENG', 'EVI', 'EVO',
       'FAH', 'FLU', 'FRE', 'FRU', 'GEN', 'GES', 'GIH', 'GLA', 'GOE',
       'GOR', 'GOS', 'GRA', 'GRE', 'GRH', 'GRO', 'GSB', 'GUE', 'GUT',
       'GVE', 'HAI', 'HLL', 'HOE', 'INNEBI', 'INNESF', 'INNRED', 'INT',
       'JUN', 'KLO', 'KOP', 'LAE', 'LAG', 'LAT', 'LEI', 'LUG', 'LUZ',
       'MAG', 'MAH', 'MAR', 'MAS', 'MER', 'MLS', 'MOA', 'MOB', 'MOE',
       'MRP', 'MTE', 'MTR', 'MUB', 'MVE', 'NABBER', 'NABDAV', 'NABDUE',
       'NABHAE', 'NABLAU', 'NABRIG', 'NABZUE', 'NAP', 'NAS', 'NEBEV',
       'NECER', 'NECOU', 'NELOC', 'NELSE', 'NEU', 'OBR', 'ORO', 'OTL',
       'PAY', '

In [3]:
batch = next(iter(loader))

In [4]:
obs_data = batch['observation_data']
num_stations = obs_data['obs'].shape[1]
key_stations = 100
query = QueryBatch(
    coords=obs_data['coords'][:, key_stations:, :],
    topo=obs_data['topo'][:, key_stations:, :],
    geo=obs_data['geo'][:, key_stations:, :],
    obs=obs_data['obs'][:, key_stations:, :],
    mask_obs=obs_data['mask_obs'][:, key_stations:, :],
)
observation_data = {
    'coords': obs_data['coords'][:, :key_stations, :],
    'topo': obs_data['topo'][:, :key_stations, :],
    'geo': obs_data['geo'][:, :key_stations, :],
    'obs': obs_data['obs'][:, :key_stations, :],
    'mask_obs': obs_data['mask_obs'][:, :key_stations, :],
}
batch['observation_data'] = observation_data
batch['query_data'] = query

In [5]:
model = OAT()

In [6]:
with torch.no_grad():
    output = model(global_data=batch['global_data'], observation_data=batch['observation_data'], query_data=batch['query_data'], )

In [ ]:

#with torch.no_grad():
# get gradients
pred, loss = model.forward_with_loss(global_data=batch['global_data'], observation_data=batch['observation_data'], query_data=batch['query_data'])
loss.backward()

In [23]:
for param in model.parameters():
    if param.grad is not None:
        print(param.grad.abs().sum())

tensor(3.3479)
tensor(0.0190)
tensor(0.0008)
tensor(0.0007)
tensor(3.2378e-05)
tensor(0.0352)
tensor(0.0016)
tensor(0.0328)
tensor(0.0028)
tensor(0.0005)
tensor(0.0006)
tensor(0.0377)
tensor(0.0018)
tensor(0.0778)
tensor(0.0029)
tensor(0.0008)
tensor(0.0007)
tensor(3.5533e-05)
tensor(0.0375)
tensor(0.0018)
tensor(0.0386)
tensor(0.0029)
tensor(0.0006)
tensor(0.0008)
tensor(0.0412)
tensor(0.0022)
tensor(0.0852)
tensor(0.0027)
tensor(6.5172)
tensor(0.0590)
tensor(0.0270)
tensor(0.0039)
tensor(0.0041)
tensor(0.0040)
tensor(0.3961)
tensor(0.0088)
tensor(0.3147)
tensor(0.0128)
tensor(0.0043)
tensor(0.0024)
tensor(0.6045)
tensor(0.0102)
tensor(1.1258)
tensor(0.0127)
tensor(0.0032)
tensor(0.0025)
tensor(0.0028)
tensor(0.2760)
tensor(0.0059)
tensor(0.3351)
tensor(0.0114)
tensor(0.0034)
tensor(0.0028)
tensor(0.5331)
tensor(0.0086)
tensor(0.9339)
tensor(0.0112)
tensor(6.5983)
tensor(0.0286)
tensor(0.0173)
tensor(0.0022)
tensor(0.0025)
tensor(0.0014)
tensor(0.3979)
tensor(0.0048)
tensor(0.4365)
te

In [29]:
output.shape

AttributeError: 'dict' object has no attribute 'shape'

In [36]:
torch.sum(batch['query_data']['obs'] == torch.nan)

tensor(0)

In [41]:
torch.sum(torch.any(pred['val_pred'] == torch.nan))

tensor(0)

In [42]:
# check if nans in output
torch.sum(output['val_pred'] == torch.nan)

tensor(0)

In [ ]:
torch.arange(10)[]

tensor([8])

In [49]:
batch['observation_data']['mask_obs'][2]

tensor([[ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True